In [2]:
# Импорт основных компонентов
from langchain_gigachat.chat_models import GigaChat
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
import os
from dotenv import load_dotenv
load_dotenv()

C:\Users\L\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
# Настройка модели GigaChat с расширенными параметрами
API_KEY = os.getenv("GIGA_KEY")
llm = GigaChat(
credentials=API_KEY,
model="GigaChat-2", # можно указать конкретную версию, по умолчанию Lite
verify_ssl_certs=False,
temperature=0.2, # настройка креативности
max_tokens=1000 # максимальная длина ответа
)
# Проверка подключения
response = llm.invoke("Привет! Как дела?")
print(response.content)

ConnectTimeout: [WinError 10060] Попытка установить соединение была безуспешной, т.к. от другого компьютера за требуемое время не получен нужный отклик, или было разорвано уже установленное соединение из-за неверного отклика уже подключенного компьютера

# СИСТЕМНОЕ И ПОЛЬЗОВАТЕЛЬСКОЕ ПРОМПТИРОВАНИЕ

In [4]:
from langchain_core.prompts import (
ChatPromptTemplate,
SystemMessagePromptTemplate,
HumanMessagePromptTemplate
)
# Шаблон системного сообщения
system_prompt = SystemMessagePromptTemplate.from_template(
"Ты — эксперт по анализу текстовых заявок на аренду жилья."
)
# Шаблон пользовательского сообщения
user_prompt = HumanMessagePromptTemplate.from_template(
"Заявка: {text}\n\nИзвлеки количество человек и верни только число."
)
# Собираем чат-шаблон
chat_prompt = ChatPromptTemplate.from_messages([system_prompt, user_prompt])

In [5]:
# При вызове цепочки:
chain = chat_prompt | llm | StrOutputParser()
result = chain.invoke({"text": "Семья из трех человек снимает квартиру"})
print(result) # ожидаем "3"

KeyboardInterrupt: 

# ПРИМЕР

In [7]:
# Простой промпт для извлечения количества людей
basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""Проанализируй текст заявки на аренду жилья и извлеки количество проживающих.
Текст заявки: {text}
Верни только число (целое число), соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.
Количество человек:"""
)

chain = basic_prompt | llm | StrOutputParser()

test_texts = [
    "Ищу квартиру для семьи из четырех человек на длительный срок"
    "Нужна студия для проживания одного человека рядом с метро"
    "Требуется двухкомнатная квартира для молодой пары"
    "Снимем жилье для троих студентов на учебный год"
    "Семья с двумя детьми ищет просторную квартиру"
]

for text in test_texts:
    result = chain.invoke({"text": text})
    print(f"Результат: {result.strip()}")

Результат: 4, 1, 2, 3, 2


# ЗАДАНИЕ

In [1]:
import pandas as pd

# Загружаем файл
df = pd.read_csv('rental_26.csv', sep=';')

# Берём первые 15 заявок
test_texts = df['text'].head(15).tolist()

# Проверяем, что загрузилось
print(f"Загружено {len(test_texts)} заявок:")
for i, text in enumerate(test_texts, 1):
    print(f"{i}. {text[:50]}...")  # показываем первые 50 символов

Загружено 15 заявок:
1. Снимем жильё с 1.09по 8.09 двухместный,су в номере...
2. Ищем недорогое жилье недалеко от моря. 3-местный и...
3. Здравствуйте,ищем жилье. 2х местный номер,с удобст...
4. Здравствуйте. Интересует жилье 3 местный номер.с20...
5. Добрый День!! Семья 4 человека, 2 взрослых, дети 1...
6. Здравствуйте. Интересует жильё в Лазаревском. 3е в...
7. Ищем жильё эконом класса, до 1000 на двоих, с 7 по...
8. Добрый день. Ищем жильё в Лазаревском. С 10.08.201...
9. Здравствуйте! Ищем жильё с 15  по 23 августа 2 взр...
10. Добрый день интересует жилье семья 5 человек двое ...
11. здравствуйте. с 24 июля по 1 августа ищем жилье дл...
12. Здравствуйте, интересует жилье п.Лазаревское 2 взр...
13. Доброе время суток) Ищем жильё- эконом, с  8 июля ...
14. Интересует жильё-эконом, кондиционер только чтобы ...
15. Здравствуйте,  ищем жилье с женой в Лазеревском, 2...


In [6]:
# Обработка всех 15 заявок
print("Обработка заявок...\n")

for i, text in enumerate(test_texts, 1):
    try:
        result = chain.invoke({"text": text})
        print(f"Заявка {i}: {result.strip()}")
    except Exception as e:
        print(f"Заявка {i}: Ошибка - {e}")

Обработка заявок...

Заявка 1: Ошибка - [WinError 10060] Попытка установить соединение была безуспешной, т.к. от другого компьютера за требуемое время не получен нужный отклик, или было разорвано уже установленное соединение из-за неверного отклика уже подключенного компьютера


KeyboardInterrupt: 